# ⚖️ Evaluación de Modelos: ¿Qué tan bueno es mi modelo?

**Módulo 03** | **Sesión 5** | **Duración estimada: 45min**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-03-modelado-predictivo/notebooks/03_02_evaluacion_modelos.ipynb)

## 🎯 Objetivos de Aprendizaje

| # | Competencia | Descripción | Nivel |
|---|-------------|-------------|-------|
| 1 | Sobreajuste vs subajuste | Comprender por qué un modelo que memoriza no es útil | Comprensión |
| 2 | Métricas de regresión | Interpretar MAE, MSE, RMSE y R² en contexto | Análisis |
| 3 | Métricas de clasificación | Construir e interpretar la matriz de confusión, accuracy, precision, recall y F1 | Análisis |
| 4 | Selección de métricas | Elegir la métrica adecuada según el problema y sus consecuencias | Evaluación |
| 5 | Validación cruzada | Comprender el concepto de k-fold cross validation | Conocimiento |

## 💡 ¿Por qué es importante para tu práctica docente?

Construir un modelo es solo la mitad del trabajo. La otra mitad --quizás la más importante-- es saber **si el modelo sirve**.

Imagina que creas un modelo para identificar estudiantes en riesgo de reprobar:

- Si el modelo dice que el 95% pasará... pero eso ya lo sabías sin modelo, **no aporta valor**
- Si el modelo identifica correctamente a los estudiantes en riesgo, **puedes intervenir a tiempo**
- Si el modelo genera muchas falsas alarmas, **desperdicias recursos** en tutorías innecesarias

Saber evaluar un modelo te permite tomar **decisiones informadas** sobre cuándo confiar en sus predicciones y cuándo ser cauteloso.

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, ConfusionMatrixDisplay
)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Librerías cargadas correctamente ✓')

---

## 🧠 Sección 1: ¿Por qué evaluar?

### El espectro sobreajuste / subajuste

Piensa en cómo estudian los estudiantes:

| Tipo de estudio | Analogía con modelos | Resultado |
|-----------------|---------------------|-----------|
| No estudia nada | **Subajuste** (underfitting) | Falla en el examen (no aprendió) |
| Memoriza las respuestas del examen pasado | **Sobreajuste** (overfitting) | Pasa el viejo, falla el nuevo |
| Entiende los conceptos | **Buen ajuste** | Generaliza a preguntas nuevas |

### ¿Cómo se manifiesta en datos?

- **Subajuste**: rendimiento malo tanto en train como en test
- **Sobreajuste**: rendimiento excelente en train, malo en test
- **Buen ajuste**: rendimiento bueno y similar en ambos

**Regla de oro**: siempre evalúa con datos que el modelo **nunca ha visto** (conjunto de test).

---

## 📊 Sección 2: Métricas de Regresión

Cuando predecimos un **número** (ej: promedio, salario, presupuesto), usamos estas métricas:

| Métrica | Qué mide | Interpretación rápida |
|---------|----------|----------------------|
| **MAE** | Error promedio absoluto | "En promedio me equivoco por X unidades" |
| **MSE** | Error promedio al cuadrado | Penaliza más los errores grandes |
| **RMSE** | Raíz del MSE | Igual que MAE pero más sensible a errores grandes |
| **R²** | Proporción de varianza explicada | "Mi modelo explica el X% de la variación" |

In [ ]:
# Ejemplo rápido con datos de rendimiento académico
df = pd.read_csv('../../datasets/universidad/rendimiento_academico.csv')
df['beca_num'] = df['beca'].astype(int)
df['trabaja_num'] = df['trabaja'].astype(int)

# Preparar datos
features = ['semestre', 'edad', 'asistencia_pct', 'materias_inscritas', 'beca_num', 'trabaja_num']
X = df[features]
y = df['promedio']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar modelo
modelo_reg = LinearRegression()
modelo_reg.fit(X_train, y_train)
y_pred = modelo_reg.predict(X_test)

# Calcular todas las métricas
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

metricas = pd.DataFrame({
    'Métrica': ['MAE', 'MSE', 'RMSE', 'R²'],
    'Valor': [mae, mse, rmse, r2],
    'Interpretación': [
        f'Error promedio: {mae:.2f} puntos de promedio',
        f'Error cuadrático: {mse:.2f} (penaliza errores grandes)',
        f'Error típico: {rmse:.2f} puntos de promedio',
        f'El modelo explica el {r2*100:.1f}% de la variación'
    ]
})

metricas

---

## 🎯 Sección 3: Métricas de Clasificación

Cuando predecimos una **categoría** (ej: aprobado/reprobado, rentable/no rentable), necesitamos métricas diferentes.

### La Matriz de Confusión

Imaginemos una **prueba médica** para detectar una enfermedad:

| | Realmente enfermo | Realmente sano |
|---|---|---|
| **Test dice: enfermo** | Verdadero Positivo (VP) | Falso Positivo (FP) |
| **Test dice: sano** | Falso Negativo (FN) | Verdadero Negativo (VN) |

- **VP (Verdadero Positivo)**: correctamente identificado como positivo
- **VN (Verdadero Negativo)**: correctamente identificado como negativo
- **FP (Falso Positivo)**: falsa alarma (decimos que sí, pero es que no)
- **FN (Falso Negativo)**: caso perdido (decimos que no, pero sí lo era)

### Las 4 métricas principales

| Métrica | Fórmula | Pregunta que responde |
|---------|---------|----------------------|
| **Accuracy** | (VP + VN) / Total | ¿Qué porcentaje acerté en total? |
| **Precision** | VP / (VP + FP) | De los que dije positivos, ¿cuántos realmente lo eran? |
| **Recall** | VP / (VP + FN) | De todos los positivos reales, ¿cuántos detecté? |
| **F1** | 2 × (P × R) / (P + R) | Balance entre precision y recall |

In [ ]:
# Ejemplo con un caso simple para entender la matriz de confusión
# Supongamos que un modelo predice si 10 estudiantes están en riesgo

y_real = np.array(     [1, 1, 1, 1, 0, 0, 0, 0, 0, 0])  # 4 en riesgo, 6 no
y_predicho = np.array( [1, 1, 0, 0, 0, 0, 0, 0, 1, 0])  # modelo predice 3 en riesgo

etiquetas_real = ['En riesgo' if v == 1 else 'Sin riesgo' for v in y_real]
etiquetas_pred = ['En riesgo' if v == 1 else 'Sin riesgo' for v in y_predicho]

resultados = pd.DataFrame({
    'Estudiante': [f'Est-{i+1}' for i in range(10)],
    'Real': etiquetas_real,
    'Predicción': etiquetas_pred,
    'Resultado': [
        'VP ✓' if r == 1 and p == 1 else
        'VN ✓' if r == 0 and p == 0 else
        'FP ✗' if r == 0 and p == 1 else
        'FN ✗'
        for r, p in zip(y_real, y_predicho)
    ]
})

print('Resultados detallados del modelo:')
print(resultados.to_string(index=False))

In [ ]:
# Visualizar la matriz de confusión
cm = confusion_matrix(y_real, y_predicho)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Sin riesgo', 'En riesgo'],
            yticklabels=['Sin riesgo', 'En riesgo'],
            annot_kws={'size': 20}, ax=ax)
ax.set_xlabel('Predicción del modelo', fontsize=13)
ax.set_ylabel('Valor real', fontsize=13)
ax.set_title('Matriz de Confusión', fontsize=14)
plt.tight_layout()
plt.show()

# Calcular métricas
print(f'Accuracy:  {accuracy_score(y_real, y_predicho):.2f} → acertó {accuracy_score(y_real, y_predicho)*100:.0f}% de las veces')
print(f'Precision: {precision_score(y_real, y_predicho):.2f} → de los que dijo "en riesgo", el {precision_score(y_real, y_predicho)*100:.0f}% realmente lo estaban')
print(f'Recall:    {recall_score(y_real, y_predicho):.2f} → de todos los que estaban en riesgo, detectó el {recall_score(y_real, y_predicho)*100:.0f}%')
print(f'F1 Score:  {f1_score(y_real, y_predicho):.2f} → balance entre precision y recall')

---

## 🤔 Sección 4: ¿Cuándo importa cada métrica?

La elección de la métrica depende del **costo de equivocarse**:

| Contexto | Tipo de error más costoso | Métrica prioritaria | Justificación |
|----------|--------------------------|--------------------|--------------|
| **Filtro de spam** | FP (email real va a spam) | **Precision** | No queremos bloquear correos importantes |
| **Diagnóstico médico** | FN (no detectar enfermedad) | **Recall** | No podemos dejar pasar un paciente enfermo |
| **Riesgo estudiantil** | FN (no detectar al que va a reprobar) | **Recall** | Preferimos dar tutoría extra a quien no la necesita |
| **Selección de becas** | FP (dar beca a quien no la necesita) | **Precision** | Los recursos son limitados |
| **Predicción general** | Ambos importan igual | **F1 Score** | Cuando no hay un error claramente peor |

### Regla práctica

- Si el costo de **no detectar** es alto → prioriza **Recall**
- Si el costo de **falsas alarmas** es alto → prioriza **Precision**
- Si ambos importan → usa **F1 Score**

---

## 🔄 Sección 5: Validación Cruzada (breve)

La división train/test tiene un problema: **depende de cómo se partieron los datos**. Si tuvimos suerte y los datos fáciles quedaron en test, el modelo parece mejor de lo que es.

### K-Fold Cross Validation

La **validación cruzada** resuelve esto:

1. Divide los datos en K partes ("folds") iguales
2. Entrena K veces: cada vez, una parte diferente es test y las demás son train
3. Promedia los K resultados

Ejemplo con K=5:

```
Ronda 1: [TEST] [train] [train] [train] [train] → R² = 0.42
Ronda 2: [train] [TEST] [train] [train] [train] → R² = 0.38
Ronda 3: [train] [train] [TEST] [train] [train] → R² = 0.45
Ronda 4: [train] [train] [train] [TEST] [train] → R² = 0.40
Ronda 5: [train] [train] [train] [train] [TEST] → R² = 0.41
                                        Promedio → R² = 0.41 ± 0.02
```

El resultado es **más confiable** porque no depende de una sola partición.

In [ ]:
# Validación cruzada rápida con nuestro modelo de rendimiento
from sklearn.model_selection import cross_val_score

# Modelo de regresión con todas las features
modelo_cv = LinearRegression()

# 5-fold cross validation
scores = cross_val_score(modelo_cv, X, y, cv=5, scoring='r2')

print('Validación Cruzada (5-fold):')
print(f'  R² por fold: {[f"{s:.4f}" for s in scores]}')
print(f'  R² promedio: {scores.mean():.4f}')
print(f'  Desv. estándar: {scores.std():.4f}')
print(f'\nInterpretación: el modelo explica en promedio el {scores.mean()*100:.1f}% ± {scores.std()*100:.1f}% de la variación')
print('Una desviación estándar baja indica que el modelo es estable')

---

## 🏛️ Sección 6: La Matriz de Confusión en Contexto FACES

Ahora apliquemos todo lo aprendido a un caso real: **predecir qué estudiantes de FACES están en riesgo académico** (promedio < 10).

Esto es exactamente el tipo de análisis que un coordinador de carrera o un decano necesita para diseñar programas de tutoría y acompañamiento.

In [ ]:
# Crear la variable objetivo binaria
df['en_riesgo'] = (df['promedio'] < 10).astype(int)

print('Distribución de la variable objetivo:')
print(df['en_riesgo'].value_counts())
print(f'\n{df["en_riesgo"].mean()*100:.1f}% de los estudiantes están en riesgo (promedio < 10)')

In [ ]:
# Preparar datos para clasificación
X_clf = df[features]
y_clf = df['en_riesgo']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)

# Entrenar regresión logística
modelo_clf = LogisticRegression(max_iter=1000, random_state=42)
modelo_clf.fit(X_train_c, y_train_c)

# Predecir
y_pred_c = modelo_clf.predict(X_test_c)

print('Modelo entrenado. Resultados en el conjunto de prueba:')
print(f'Total de estudiantes en test: {len(y_test_c)}')
print(f'Predicciones "en riesgo": {y_pred_c.sum()}')
print(f'Realmente en riesgo: {y_test_c.sum()}')

In [ ]:
# Matriz de confusión visual
fig, ax = plt.subplots(figsize=(7, 6))
cm_faces = confusion_matrix(y_test_c, y_pred_c)
sns.heatmap(cm_faces, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['Sin riesgo', 'En riesgo'],
            yticklabels=['Sin riesgo', 'En riesgo'],
            annot_kws={'size': 20}, ax=ax)
ax.set_xlabel('Predicción del modelo', fontsize=13)
ax.set_ylabel('Valor real', fontsize=13)
ax.set_title('Matriz de Confusión — Riesgo Estudiantil FACES', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Reporte completo de clasificación
print('Reporte de Clasificación:')
print('=' * 55)
print(classification_report(y_test_c, y_pred_c, 
                            target_names=['Sin riesgo', 'En riesgo']))

# Interpretación en contexto
prec = precision_score(y_test_c, y_pred_c)
rec = recall_score(y_test_c, y_pred_c)

print('Interpretación para el coordinador:')
print(f'  Precision = {prec:.2f} → De cada 100 estudiantes que el modelo señala como')
print(f'    "en riesgo", {prec*100:.0f} realmente lo están')
print(f'  Recall = {rec:.2f} → De cada 100 estudiantes que realmente están en riesgo,')
print(f'    el modelo detecta a {rec*100:.0f} de ellos')

---

## ✏️ Ejercicios

### Ejercicio 1: Cálculo manual de métricas

Dada la siguiente matriz de confusión de un modelo que predice si un estudiante aprueba o reprueba:

| | Pred: Aprueba | Pred: Reprueba |
|---|---|---|
| **Real: Aprueba** | 85 | 5 |
| **Real: Reprueba** | 15 | 20 |

Calcula a mano (puedes usar Python como calculadora):
1. Accuracy
2. Precision (para la clase "Reprueba")
3. Recall (para la clase "Reprueba")
4. F1 Score (para la clase "Reprueba")

In [ ]:
# Ejercicio 1: Cálculo manual de métricas
# Datos de la matriz de confusión
VP = 20   # Verdaderos Positivos (correctamente identificados como "Reprueba")
VN = 85   # Verdaderos Negativos (correctamente identificados como "Aprueba")
FP = 5    # Falsos Positivos (dijo Reprueba, pero era Aprueba)
FN = 15   # Falsos Negativos (dijo Aprueba, pero era Reprueba)

# Tu código aquí
# accuracy = ...
# precision = ...
# recall = ...
# f1 = ...


### Ejercicio 2: Selección de métricas

Imagina que eres coordinador de la carrera de Economía y estás construyendo un modelo para predecir deserción estudiantil (si un estudiante abandonará la carrera).

1. ¿Qué métrica priorizarías: precision o recall? Justifica tu respuesta.
2. ¿Cuál sería el costo de un Falso Negativo (no detectar a un desertor)?
3. ¿Cuál sería el costo de un Falso Positivo (señalar como desertor a alguien que no lo es)?

Escribe tu respuesta en la celda de abajo (puedes usar Markdown o comentarios en Python).

In [ ]:
# Ejercicio 2: Selección de métricas para predicción de deserción
# Tu código aquí

# 1. ¿Qué métrica priorizarías?
# Respuesta: 

# 2. ¿Cuál es el costo de un Falso Negativo?
# Respuesta: 

# 3. ¿Cuál es el costo de un Falso Positivo?
# Respuesta: 


---

## 📋 Resumen

| Concepto | Punto clave |
|----------|-------------|
| **Sobreajuste** | El modelo memoriza los datos de entrenamiento y no generaliza |
| **MAE / RMSE** | Miden el error en las mismas unidades de la variable (para regresión) |
| **R²** | Proporción de variación explicada (0 a 1, más alto es mejor) |
| **Matriz de confusión** | Tabla que muestra VP, VN, FP, FN de una clasificación |
| **Accuracy** | Porcentaje total de aciertos (puede engañar con datos desbalanceados) |
| **Precision** | De los positivos predichos, ¿cuántos lo son realmente? |
| **Recall** | De los positivos reales, ¿cuántos detectó el modelo? |
| **F1 Score** | Media armónica entre precision y recall |
| **Cross Validation** | Evalúa el modelo K veces con diferentes particiones (más robusto) |

## 📚 Referencias

1. James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning* (2nd ed.). Springer. Capítulo 5: Resampling Methods.

2. Scikit-learn developers. (2024). *Model evaluation: quantifying the quality of predictions*. https://scikit-learn.org/stable/modules/model_evaluation.html

3. Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly. Capítulo 3: Classification.

4. Powers, D. M. W. (2011). *Evaluation: From precision, recall and F-measure to ROC, informedness, markedness and correlation*. Journal of Machine Learning Technologies, 2(1), 37-63.